In [1]:
pip install torch-geometric torch-cluster torch-scatter torch-sparse

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Users\Dell\AppData\Local\Programs\Python\Python314\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\Dell\AppData\Local\Temp\pip-build-env-8utw3ao6\overlay\Lib\site-packages\

In [2]:
pip install scipy

   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.0/37.3 MB 6.2 MB/s eta 0:00:06
   - -------------------------------------- 1.8/37.3 MB 4.7 MB/s eta 0:00:08
   -- ------------------------------------- 2.6/37.3 MB 4.4 MB/s eta 0:00:08
   --- ------------------------------------ 3.4/37.3 MB 4.3 MB/s eta 0:00:08
   ---- ----------------------------------- 4.2/37.3 MB 4.1 MB/s eta 0:00:08
   ----- ---------------------------------- 5.0/37.3 MB 4.1 MB/s eta 0:00:08
   ------ --------------------------------- 5.8/37.3 MB 4.1 MB/s eta 0:00:08
   ------- -------------------------------- 6.8/37.3 MB 4.0 MB/s eta 0:00:08
   ------- -------------------------------- 7.3/37.3 MB 4.0 MB/s eta 0:00:08
   -------- ------------------------------- 8.4/37.3 MB 4.0 MB/s eta 0:00:08
   --------- ------------------------------ 9.2/37.3 MB 4.0 MB/s eta 0:00:08
   ---------- ----------------------------- 10.0/37.3 MB 4.0 MB/s eta 0:00:07
   --


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# 1. Install ONLY the core library first
%pip install torch-geometric

# 2. Install scipy
%pip install scipy

  Using cached torch_geometric-2.7.0-py3-none-any.whl.metadata (63 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------------------------------- ------- 1.0/1.3 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 4.4 MB/s  0:00:00

   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   -- -------------------------------------  1/16 [urllib3]
   ----- ----------------------------------  2/16 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import h5py
import torch
import numpy as np
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader as GNNLoader
from scipy.spatial import KDTree

class JetGraphDataset(Dataset):
    def __init__(self, file_path, num_samples=5000):
        super().__init__()
        with h5py.File(file_path, 'r') as f:
            # Load images and labels
            self.images = f['X_jets'][:num_samples]
            self.labels = f['y'][:num_samples]
            
    def len(self):
        return len(self.images)

    def get(self, idx):
        img = self.images[idx] 
        # FIX 1: Force label to be a Python integer immediately
        label = int(self.labels[idx])

        coords = np.argwhere(np.sum(img, axis=-1) > 0)
        
        if len(coords) == 0:
            return Data(x=torch.zeros((1, 5)), 
                        edge_index=torch.empty((2, 0), dtype=torch.long), 
                        y=torch.tensor([label], dtype=torch.long))

        raw_hits = img[coords[:, 0], coords[:, 1]] 
        norm_coords = coords.astype(np.float32) / 125.0 
        
        x_np = np.hstack([norm_coords, raw_hits.astype(np.float32)])
        x = torch.from_numpy(x_np).float()
        
        tree = KDTree(coords)
        _, indices = tree.query(coords, k=7)
        
        # FIX 2: Force row and col to be explicitly 64-bit integers
        row = np.repeat(np.arange(len(coords), dtype=np.int64), 6)
        col = indices[:, 1:].flatten().astype(np.int64)
        
        edge_index = torch.tensor(np.stack([row, col]), dtype=torch.long)

        return Data(x=x, edge_index=edge_index, y=torch.tensor([label], dtype=torch.long))
# Initialize
file_path = 'quark-gluon_data-set_n139306.hdf5'
graph_dataset = JetGraphDataset(file_path, num_samples=2000)
train_loader_gnn = GNNLoader(graph_dataset, batch_size=32, shuffle=True)

# Test run: If this prints, you are ready to train!
print(f"Sample 0 Graph: {graph_dataset[0]}")

Sample 0 Graph: Data(x=[884, 5], edge_index=[2, 5304], y=[1])


In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.optim as optim

# 1. Define the Graph Neural Network Architecture
class JetGNN(torch.nn.Module):
    def __init__(self):
        super(JetGNN, self).__init__()
        # Input: 5 features [x, y, Track, ECAL, HCAL]
        self.conv1 = GCNConv(5, 32)
        self.conv2 = GCNConv(32, 64)
        self.conv3 = GCNConv(64, 64)
        
        # Classification layer (Quark vs Gluon = 2 classes)
        self.fc = torch.nn.Linear(64, 2)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Graph Convolution Layers with ReLU activation
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))

        # Global Mean Pooling: Collapses the graph nodes into a single vector
        x = global_mean_pool(x, batch)

        # Output Log-Softmax for classification
        return F.log_softmax(self.fc(x), dim=1)

# 2. Initialize Model, Optimizer, and Loss
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gnn = JetGNN().to(device)
optimizer = optim.Adam(model_gnn.parameters(), lr=0.001)

# 3. Define the Training Function
def train_gnn(epochs=10):
    model_gnn.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        for data in train_loader_gnn: # This is the loader we created with SciPy
            data = data.to(device)
            optimizer.zero_grad()
            out = model_gnn(data)
            
            # Loss and Backprop
            loss = F.nll_loss(out, data.y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Calculate Accuracy for monitoring
            pred = out.argmax(dim=1)
            correct += int((pred == data.y).sum())
            total += data.y.size(0)
            
        avg_loss = total_loss / len(train_loader_gnn)
        accuracy = correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {accuracy:.4f}")

# 4. Start Training!
train_gnn(epochs=10)

Epoch 1/10 | Loss: 0.6936 | Acc: 0.5125
Epoch 2/10 | Loss: 0.6935 | Acc: 0.4945
Epoch 3/10 | Loss: 0.6929 | Acc: 0.5125
Epoch 4/10 | Loss: 0.6930 | Acc: 0.5125
Epoch 5/10 | Loss: 0.6929 | Acc: 0.5125
Epoch 6/10 | Loss: 0.6934 | Acc: 0.5125
Epoch 7/10 | Loss: 0.6929 | Acc: 0.5125
Epoch 8/10 | Loss: 0.6928 | Acc: 0.5125
Epoch 9/10 | Loss: 0.6928 | Acc: 0.5125
Epoch 10/10 | Loss: 0.6928 | Acc: 0.5125
